# SqlEventsDelta

[Codility task](https://app.codility.com/programmers/trainings/6/sql_events_delta/) - Exercise 6, SQL - *easy*

Given

```sql
create table events (
    event_type integer not null,
    value integer not null,
    time timestamp not null,
    unique(event_type, time)
);
```

return, for each `event_type` registered **more than once**, the difference between the latest value and the second latest value (by `time`), ordered by `event_type`.

Expected for the example data: `(2, -5)` and `(3, 4)`.

The SQL lives in `solution.sql` (the file you paste into Codility). This notebook loads `setup.sql` into an in-memory **SQLite** database, runs the solution and checks it.

In [ ]:
import sqlite3
from pathlib import Path

HERE = Path.cwd()          # Jupyter starts the kernel in the notebook's folder


def fresh_db(with_example=True):
    """New in-memory SQLite database; optionally loaded with setup.sql (tables + example data)."""
    conn = sqlite3.connect(":memory:")
    if with_example:
        conn.executescript((HERE / "setup.sql").read_text(encoding="utf-8"))
    return conn


def run(conn, sql_file="solution.sql"):
    """Execute the single query stored in `sql_file` and return all its rows."""
    return conn.execute((HERE / sql_file).read_text(encoding="utf-8")).fetchall()


def show(rows, headers):
    """Tiny table printer (no pandas needed)."""
    table = [headers] + [[str(c) for c in row] for row in rows]
    widths = [max(len(r[i]) for r in table) for i in range(len(headers))]
    for n, row in enumerate(table):
        print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))
        if n == 0:
            print("-+-".join("-" * w for w in widths))

## The solution

In [ ]:
print((HERE / "solution.sql").read_text(encoding="utf-8"))

## How it works

1. `row_number() over (partition by event_type order by time desc)` numbers the rows of each type from the most recent (`1`) to the oldest.
2. `where rn <= 2` keeps the latest two rows of every type.
3. `case when rn = 1 then value else -value end`, summed per type, is `latest - second latest`.
4. `having count(*) = 2` drops the types that were registered only once.

## Official example

In [ ]:
conn = fresh_db()
rows = run(conn)
show(rows, ["event_type", "value"])
assert rows == [(2, -5), (3, 4)]
print("\nexample OK")

## Edge cases

Each case starts from an empty `events` table and inserts its own rows.

In [ ]:
def delta(rows):
    """rows: iterable of (event_type, value, time). Returns the SQL result."""
    conn = fresh_db()
    conn.execute("delete from events")
    conn.executemany("insert into events (event_type, value, time) values (?, ?, ?)", rows)
    return run(conn)


# no rows at all, and only single events
assert delta([]) == []
assert delta([(1, 10, "2020-01-01 00:00:00"), (2, 20, "2020-01-01 00:00:00")]) == []

# exactly two events: latest minus earliest, whatever the insertion order
assert delta([(1, 3, "2020-01-02 00:00:00"), (1, 10, "2020-01-01 00:00:00")]) == [(1, -7)]
assert delta([(1, 10, "2020-01-01 00:00:00"), (1, 3, "2020-01-02 00:00:00")]) == [(1, -7)]

# three events: only the latest two count (3 - 100, the 5 is ignored)
assert delta([
    (1, 5, "2020-01-01 00:00:00"),
    (1, 100, "2020-01-02 00:00:00"),
    (1, 3, "2020-01-03 00:00:00"),
]) == [(1, -97)]

# negative values and the same timestamp on different event types
assert delta([
    (7, -4, "2020-05-05 10:00:00"), (7, -10, "2020-05-05 09:00:00"),
    (3, 1, "2020-05-05 10:00:00"), (3, 2, "2020-05-05 09:00:00"),
]) == [(3, -1), (7, 6)]

# ordering: ascending by event_type even when inserted in reverse
rows = delta([(t, 1, "2020-01-02 00:00:00") for t in (9, 5, 1)] + [(t, 0, "2020-01-01 00:00:00") for t in (9, 5, 1)])
assert [r[0] for r in rows] == [1, 5, 9]
print("edge cases OK")

## Randomised check against a Python reference

The reference groups the rows by type, sorts each group by time and subtracts the last two values.

In [ ]:
import random
from collections import defaultdict
from datetime import datetime, timedelta


def reference(rows):
    groups = defaultdict(list)
    for event_type, value, time in rows:
        groups[event_type].append((time, value))
    out = []
    for event_type in sorted(groups):
        events = sorted(groups[event_type])
        if len(events) >= 2:
            out.append((event_type, events[-1][1] - events[-2][1]))
    return out


rng = random.Random(5)
start = datetime(2015, 5, 9, 12, 0, 0)
for _ in range(300):
    rows = []
    for event_type in rng.sample(range(1, 20), rng.randint(1, 6)):
        minutes = rng.sample(range(0, 600), rng.randint(1, 6))        # distinct times per type
        for m in minutes:
            time = (start + timedelta(minutes=m)).strftime("%Y-%m-%d %H:%M:%S")
            rows.append((event_type, rng.randint(-50, 50), time))
    rng.shuffle(rows)
    assert delta(rows) == reference(rows), rows
print("SQL result equals the Python reference on 300 random tables")

## Notes

- Window functions (`row_number`) need SQLite 3.25+ (any Python 3.8+ bundles one) and are standard in PostgreSQL.
- `unique(event_type, time)` guarantees there are no ties for "latest", so `row_number` is deterministic here. With possible ties you would have to pick a tie-breaker.
- Alternative without window functions: join each row with the maximum `time` of its type, then repeat for the maximum below it. It is longer and easier to get wrong.
- ISO-formatted timestamps sort correctly even when stored as text (SQLite); in PostgreSQL `timestamp` is a real type.